In [1]:
import os
import datasets
from datasets import load_dataset

dataset_path = 'datasets'
dataset_name = 'hazemessam/otto-recsys'

cache_dir = 'cache'

os.makedirs(dataset_path, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

dataset = load_dataset(dataset_name, cache_dir=cache_dir)

In [2]:
dataset['train'].shape[0]

12899779

In [ ]:
import torch
from tqdm import tqdm
from pathlib import Path
from joblib import Parallel, delayed
from torch.nn import functional as F
from torch.utils.data import Dataset

class RecDataset(Dataset):
    def __init__(
            self, 
            hf_dataset: datasets.dataset_dict.DatasetDict, 
            window_len: int,
            cache_dir: str | None = None
        ):
        self.pad_id = 0
        self.mask_id = 1
        self.unk_id = 2
        self.window_len = window_len

        self.data = self.load_or_process(hf_dataset, cache_dir)
            
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

    def load_or_process(self, dataset, cache_dir):
        if cache_dir is None:
            return self.sample_dataset(dataset)

        path = Path(cache_dir) / f"windows_{self.window_len}.pt"

        if path.exists():
            print(f"Loading cache: {path}")
            return torch.load(path, weights_only=True)

        data = self.sample_dataset(dataset)

        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(data, path)

        return data

    def sampling_session(self, session: dict):
        ids = []

        for event in session['events']:
            if event['type'] != 'clicks':
                continue

            ids.append(event['aid'] + max([self.unk_id, self.pad_id, self.mask_id]))

        ids = torch.tensor(ids)

        dim = 0
        step = 1
        
        if ids.shape[dim] < self.window_len:
            return F.pad(ids, (0, self.window_len - ids.shape[dim]), 'constant', self.pad_id).unfold(dim, self.window_len, step)
        else:
            return ids.unfold(dim, self.window_len, step) 

    def sample_dataset(self, dataset):
        samples = []

        p = Parallel(n_jobs=-1, batch_size="auto")
        samples = p(delayed(self.sampling_session)(session) for session in tqdm(dataset))

        return torch.concat(samples, dim=0)

In [4]:
window_len = 5

train_ds_path = 'cache/train/'
test_ds_path = 'cache/test/'

train_ds = RecDataset(dataset['train'], window_len, train_ds_path)
test_ds = RecDataset(dataset['test'], window_len, test_ds_path)

100%|██████████| 1671803/1671803 [03:35<00:00, 7760.28it/s]


In [11]:
from torch.utils.data import random_split, DataLoader

val_size = 0.1

train, val = random_split(train_ds, [1-val_size, val_size])
test = test_ds

train_loader = DataLoader(
    train,
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    val,
    batch_size=64,
    shuffle=False
)

In [ ]:
def mask_batch(batch):
    mask = (batch != 0).sum(dim=1)
    mask_positions = (torch.rand(mask.shape) * mask).long()

    batch_size = batch.shape[0]
    
    targets = batch[torch.arange(0, batch_size), mask_positions]

    input_ids = batch.clone()   
    input_ids[torch.arange(0, batch_size), mask_positions] = 1

    return input_ids, targets, mask_positions